<p align='center'>
<a href='https://github.com/bono-p/minillm_v2'><img src='https://img.shields.io/badge/GitHub-bono--p%2Fminillm__v2-181717?style=for-the-badge&logo=github'/></a>
&nbsp;
<a href='https://colab.research.google.com/github/bono-p/minillm_v2/blob/main/MiniLLM_v2_DevLab.ipynb'><img src='https://colab.research.google.com/assets/colab-badge.svg' height='28'/></a>
</p>

# 🧠 MiniLLM v2 — DevLab

Transformer decoder-only moderne, entraînable sur GPU Colab gratuit (T4).
Architecture : **RoPE · RMSNorm · SwiGLU · GQA · Flash Attention**

---

## 🗺️ Plan

| # | Section | Description |
|---|---------|-------------|
| **0** | Variables globales | Chemins Drive définis une seule fois |
| **1** | Setup | GPU · Drive · Dépendances · Repo GitHub |
| **2** | Architecture | Choisir et inspecter la taille du modèle |
| **3** | Datasets | Wikipedia FR · PIAF QA (HuggingFace) |
| **4** | Tokenisation | Préparer les `.bin` d'entraînement (dtype int32) |
| **5** | Pré-entraînement | Entraîner sur Wikipedia FR |
| **6** | Fine-tuning QA | Spécialiser sur PIAF |
| **7** | Génération | Tester le modèle |
| **8** | Sauvegarde | Drive · Rolling checkpoints · Export |

> ⚡ **GPU requis** : `Exécution > Modifier le type d'exécution > GPU (T4)`  
> ▶️ Exécute les cellules **dans l'ordre** de haut en bas.


---
## 0️⃣ Variables globales
> Définis ici tous tes chemins Drive. Le reste du notebook les utilise automatiquement.


In [1]:
# @title ⚙️ 0.1 — Chemins et paramètres globaux
# ── Personalise ici si besoin ──────────────────────────
GITHUB_USER   = 'bono-p'              # @param {type:'string'}
GITHUB_REPO   = 'minillm_v2'         # @param {type:'string'}
DRIVE_BASE    = '/content/drive/MyDrive/MiniLLM_v2'  # @param {type:'string'}
MODEL_SIZE    = '50M'                 # @param ['15M','50M','125M','350M']
ROLLING_KEEP  = 2                     # @param {type:'slider',min:1,max:5,step:1}
# ───────────────────────────────────────────────────────

import os

REPO_DIR       = f'/content/{GITHUB_REPO}'
DATA_DIR       = f'{DRIVE_BASE}/data'
PRETRAIN_BIN   = f'{DATA_DIR}/pretrain'
FINETUNE_BIN   = f'{DATA_DIR}/finetune'
CKPT_PRETRAIN  = f'{DRIVE_BASE}/checkpoints/pretrain'
CKPT_FINETUNE  = f'{DRIVE_BASE}/checkpoints/finetune'
WIKI_CACHE     = None   # défini à la section 3.1
PIAF_CACHE     = f'{DATA_DIR}/piaf_qa.json'
PRETRAIN_TRAIN = f'{PRETRAIN_BIN}/train.bin'
PRETRAIN_VAL   = f'{PRETRAIN_BIN}/val.bin'
FINETUNE_TRAIN = f'{FINETUNE_BIN}/train.bin'
FINETUNE_VAL   = f'{FINETUNE_BIN}/val.bin'

print('✅ Variables globales définies')
print(f'  Drive base   : {DRIVE_BASE}')
print(f'  Modèle       : MiniLLM-{MODEL_SIZE}')
print(f'  Rolling keep : {ROLLING_KEEP} checkpoints numérotés max')


✅ Variables globales définies
  Drive base   : /content/drive/MyDrive/MiniLLM_v2
  Modèle       : MiniLLM-50M
  Rolling keep : 2 checkpoints numérotés max


---
## 1️⃣ Setup


In [2]:
# @title 🖥️ 1.1 — Vérification GPU
import subprocess, torch

res = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True
)
if res.returncode == 0:
    print(f'✅ GPU : {res.stdout.strip()}')
else:
    print('❌ Aucun GPU détecté !')
    print('   → Exécution > Modifier le type d\'exécution > GPU (T4)')
    raise SystemExit('GPU requis pour entraîner MiniLLM v2')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'BF16     : {torch.cuda.is_bf16_supported()}')
DEVICE = 'cuda'


✅ GPU : Tesla T4, 15360 MiB, 14913 MiB
PyTorch  : 2.11.0+cu128
CUDA     : 12.8
BF16     : True


In [3]:
# @title 📦 1.2 — Installation des dépendances
%%capture cap
import subprocess
subprocess.run(['pip','install','tiktoken','datasets','-q'], check=True)

import tiktoken, datasets, numpy, torch
print(f'✅ tiktoken  {tiktoken.__version__}')
print(f'✅ datasets  {datasets.__version__}')
print(f'✅ numpy     {numpy.__version__}')
print(f'✅ torch     {torch.__version__}')


In [4]:
# @title 💾 1.3 — Montage Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Créer tous les dossiers nécessaires
for d in [DATA_DIR, PRETRAIN_BIN, FINETUNE_BIN, CKPT_PRETRAIN, CKPT_FINETUNE]:
    os.makedirs(d, exist_ok=True)

print('✅ Google Drive monté')
print(f'  📁 Données          : {DATA_DIR}')
print(f'  📁 Ckpt pré-entr.   : {CKPT_PRETRAIN}')
print(f'  📁 Ckpt fine-tuning : {CKPT_FINETUNE}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive monté
  📁 Données          : /content/drive/MyDrive/MiniLLM_v2/data
  📁 Ckpt pré-entr.   : /content/drive/MyDrive/MiniLLM_v2/checkpoints/pretrain
  📁 Ckpt fine-tuning : /content/drive/MyDrive/MiniLLM_v2/checkpoints/finetune


In [5]:
# @title 📂 1.4 — Clonage du repo GitHub
import os, sys

if os.path.exists(REPO_DIR):
    print('Mise à jour du repo...')
    !cd {REPO_DIR} && git pull -q
else:
    url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
    print(f'Clonage depuis {url}...')
    !git clone {url} {REPO_DIR} -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'✅ Repo prêt dans {REPO_DIR}')
py_files = sorted(f for f in os.listdir('.') if f.endswith('.py'))
print(f'   Fichiers Python : {py_files}')


Mise à jour du repo...
✅ Repo prêt dans /content/minillm_v2
   Fichiers Python : ['config.py', 'generate.py', 'inspect_model.py', 'model.py', 'prepare_data.py', 'push_to_github.py', 'train.py']


---
## 2️⃣ Architecture du modèle
> Choisis la taille adaptée à ton GPU. Pour Colab T4 (15 GB VRAM) : **15M ou 50M** recommandé.


In [6]:
# @title 📊 2.1 — Tableau comparatif des presets
from config import PRESETS

print(f'  {"Size":>6} | {"Params":>7} | {"Layers":>6} | {"d_model":>7} | '
      f'{"Heads Q":>7} | {"KV":>4} | {"FFN":>5} | {"Ctx":>5} | GPU')
print('  ' + '─'*70)
for name, cfg in PRESETS.items():
    n    = cfg.count_params()
    typ  = 'MHA' if cfg.kv_heads == cfg.n_heads else f'GQA×{cfg.n_heads//cfg.kv_heads}'
    vram = n*4*4/1e9
    gpu  = '✅ T4' if vram < 13 else ('⚠️ Pro' if vram < 40 else '🔴 A100')
    print(f'  {name:>6} | {n/1e6:>5.1f}M | {cfg.n_layers:>6} | {cfg.d_model:>7} | '
          f'{cfg.n_heads:>7} | {typ:>4} | {cfg.ffn_hidden:>5} | {cfg.max_seq_len:>5} | {gpu}')


    Size |  Params | Layers | d_model | Heads Q |   KV |   FFN |   Ctx | GPU
  ──────────────────────────────────────────────────────────────────────
     15M |  29.9M |      6 |     384 |       6 |  MHA |  1024 |  1024 | ✅ T4
     50M |  57.9M |     10 |     512 |       8 |  MHA |  1408 |  1024 | ✅ T4
    125M | 123.6M |     12 |     768 |      12 |  MHA |  2048 |  1024 | ✅ T4
    350M | 329.9M |     24 |    1024 |      16 | GQA×2 |  2752 |  2048 | ✅ T4
      1B | 989.1M |     20 |    2048 |      16 | GQA×4 |  5504 |  4096 | ⚠️ Pro


In [7]:
# @title 🔬 2.2 — Inspecter le modèle choisi
import torch, math, sys
sys.path.insert(0, REPO_DIR)
from config import PRESETS
from model  import MiniLLM

cfg   = PRESETS[MODEL_SIZE]
model = MiniLLM(cfg)
n     = model.n_params
n_emb = cfg.vocab_size * cfg.d_model
n_blk = sum(p.numel() for nm,p in model.named_parameters() if 'blocks' in nm)

print(f'{'═'*52}')
print(f'  MiniLLM-{MODEL_SIZE}  ({n/1e6:.2f}M paramètres)')
print(f'{'═'*52}')
print(f'  Embedding  : {n_emb/1e6:.2f}M  ({n_emb/n*100:.0f}%)')
print(f'  Blocs ×{cfg.n_layers}   : {n_blk/1e6:.2f}M  ({n_blk/n*100:.0f}%)')
print(f'  LM Head    : {"partagé (tie_embeddings)" if cfg.tie_embeddings else "séparé"}')
print(f'  VRAM entr  : ~{n*4*4/1e9:.2f} GB')
print()

x = torch.randint(0, cfg.vocab_size, (2, 64))
with torch.no_grad():
    logits, loss = model(x, x)
expected = math.log(cfg.vocab_size)
print(f'  ✅ Forward OK  |  Loss initiale : {loss.item():.3f}  (attendu ≈ {expected:.2f})')
print(f'{'═'*52}')


════════════════════════════════════════════════════
  MiniLLM-50M  (57.85M paramètres)
════════════════════════════════════════════════════
  Embedding  : 25.73M  (44%)
  Blocs ×10   : 32.12M  (56%)
  LM Head    : partagé (tie_embeddings)
  VRAM entr  : ~0.93 GB

  ✅ Forward OK  |  Loss initiale : 9.303  (attendu ≈ 10.82)
════════════════════════════════════════════════════


---
## 3️⃣ Téléchargement des datasets
Les fichiers sont mis en **cache sur Drive** → pas re-téléchargés si tu relances.

| Dataset | Source | Usage |
|---------|--------|-------|
| **Wikipedia FR** | `wikimedia/wikipedia` (HuggingFace) | Pré-entraînement |
| **PIAF** | `etalab-ia/piaf` (HuggingFace) | Fine-tuning QA |


In [8]:
# @title 📰 3.1 — Wikipedia français (pré-entraînement)
import os, sys
from datasets import load_dataset

WIKI_FRACTION = '2%'  # @param ['1%','2%','5%','10%','20%','100%']

WIKI_CACHE = f'{DATA_DIR}/wikipedia_fr_{WIKI_FRACTION.replace("%","p")}.txt'

if os.path.exists(WIKI_CACHE) and os.path.getsize(WIKI_CACHE) > 1000:
    size_mb = os.path.getsize(WIKI_CACHE)/1e6
    print(f'✅ Cache Drive : {WIKI_CACHE}  ({size_mb:.0f} MB) — pas de re-téléchargement')
else:
    print(f'Téléchargement Wikipedia FR ({WIKI_FRACTION})...')
    # Identifiant correct HuggingFace 2024 : wikimedia/wikipedia
    wiki = load_dataset(
        'wikimedia/wikipedia',
        '20231101.fr',
        split=f'train[:{WIKI_FRACTION}]',
        trust_remote_code=False,
    )
    print(f'  {len(wiki):,} articles chargés — écriture vers Drive...')
    chars = 0
    with open(WIKI_CACHE, 'w', encoding='utf-8') as f:
        for i, art in enumerate(wiki):
            text = art['title'] + '\n\n' + art['text'] + '\n\n'
            f.write(text)
            chars += len(text)
            if (i+1) % 5000 == 0:
                print(f'  {i+1:,} articles — {chars/1e6:.0f} MB', end='\r')
    size_mb = os.path.getsize(WIKI_CACHE)/1e6
    print(f'\n✅ Sauvegardé : {WIKI_CACHE}  ({size_mb:.0f} MB)')

# Aperçu
with open(WIKI_CACHE, encoding='utf-8') as f:
    preview = f.read(350)
print(f'\nAperçu :\n{"─"*44}\n{preview}\n{"─"*44}')


✅ Cache Drive : /content/drive/MyDrive/MiniLLM_v2/data/wikipedia_fr_2p.txt  (630 MB) — pas de re-téléchargement

Aperçu :
────────────────────────────────────────────
Antoine Meillet

Antoine Meillet, né le  à Moulins (Allier) et mort le  à Châteaumeillant (Cher), est un philologue français, le principal linguiste français des premières décennies du .

Biographie

Enfance et formation 
Paul Jules Antoine Meillet est d'origine bourbonnaise, fils d'un notaire de Châteaumeillant (Cher). Il naît à Moulins le 11 nove
────────────────────────────────────────────


In [9]:
# @title ❓ 3.2 — PIAF : dataset QA français (3 835 paires)
import os, json
from datasets import load_dataset

if os.path.exists(PIAF_CACHE) and os.path.getsize(PIAF_CACHE) > 100:
    with open(PIAF_CACHE, encoding='utf-8') as f:
        piaf_data = json.load(f)
    print(f'✅ Cache Drive : {PIAF_CACHE}  ({len(piaf_data)} exemples)')
else:
    print('Téléchargement PIAF (etalab-ia/piaf)...')
    piaf = load_dataset('etalab-ia/piaf', split='train', trust_remote_code=False)
    piaf_data = []
    for ex in piaf:
        answers = ex['answers']['text']
        if not answers:
            continue
        piaf_data.append({
            'context':  ex['context'][:800],
            'question': ex['question'],
            'answer':   answers[0],
        })
    with open(PIAF_CACHE, 'w', encoding='utf-8') as f:
        json.dump(piaf_data, f, ensure_ascii=False, indent=1)
    print(f'✅ PIAF sauvegardé : {len(piaf_data)} exemples → {PIAF_CACHE}')

# Aperçu de 2 exemples
print(f'\n── Exemples PIAF ──────────────────────────────')
for ex in piaf_data[:2]:
    print(f'  Q : {ex["question"]}')
    print(f'  R : {ex["answer"]}')
    print()


✅ Cache Drive : /content/drive/MyDrive/MiniLLM_v2/data/piaf_qa.json  (3835 exemples)

── Exemples PIAF ──────────────────────────────
  Q : Combien de personnes travaillent au ministère des sports
  R : 100 000

  Q : Combien d'employeurs
  R : 20 000



---
## 4️⃣ Tokenisation
> Conversion texte → tokens → fichiers `.bin` en **int32**.

> ⚠️ **Pourquoi int32 ?**  
> cl100k_base a 100 277 tokens → uint16 max = 65 535 → **overflow pour les tokens > 65k**.  
> int32 gère jusqu'à 2 milliards → aucun problème.


In [10]:
# @title 🔡 4.1 — Tokeniser Wikipedia FR (pré-entraînement)
import os, sys, numpy as np, tiktoken
sys.path.insert(0, REPO_DIR)

# Vérifier si déjà tokenisé et valide
if (os.path.exists(PRETRAIN_TRAIN) and os.path.getsize(PRETRAIN_TRAIN) > 0 and
    os.path.exists(PRETRAIN_VAL)   and os.path.getsize(PRETRAIN_VAL)   > 0):
    n_tr = len(np.memmap(PRETRAIN_TRAIN, dtype=np.int32, mode='r'))
    n_vl = len(np.memmap(PRETRAIN_VAL,   dtype=np.int32, mode='r'))
    print(f'✅ Déjà tokenisé : train={n_tr:,} | val={n_vl:,} tokens (int32)')
else:
    assert WIKI_CACHE and os.path.exists(WIKI_CACHE), (
        'WIKI_CACHE introuvable. Lance la cellule 3.1 d\'abord.'
    )
    print('Tokenisation Wikipedia FR (streaming, économise la RAM)...')
    enc       = tiktoken.get_encoding('cl100k_base')
    val_ratio = 0.01
    import random; rng = random.Random(42)
    total_tr = total_vl = 0
    current  = []

    with open(WIKI_CACHE, 'r', encoding='utf-8') as fin,\
         open(PRETRAIN_TRAIN, 'wb') as ftr,\
         open(PRETRAIN_VAL,   'wb') as fvl:

        def flush(lines):
            global total_tr, total_vl
            text = ''.join(lines).strip()
            if not text: return
            toks = enc.encode(text)
            if not toks: return
            arr = np.array(toks, dtype=np.int32)
            if rng.random() < val_ratio:
                fvl.write(arr.tobytes()); total_vl += len(arr)
            else:
                ftr.write(arr.tobytes()); total_tr += len(arr)

        for line in fin:
            if line.strip() == '':
                flush(current); current = []
            else:
                current.append(line)
        flush(current)   # dernier article

    n_tr = len(np.memmap(PRETRAIN_TRAIN, dtype=np.int32, mode='r'))
    n_vl = len(np.memmap(PRETRAIN_VAL,   dtype=np.int32, mode='r'))
    print(f'✅ Tokenisé : train={n_tr:,} | val={n_vl:,} tokens (int32)')


✅ Déjà tokenisé : train=335,028,176 | val=3,388,894 tokens (int32)


In [11]:
# @title 🔡 4.2 — Tokeniser PIAF (fine-tuning QA)
import os, json, sys, numpy as np, tiktoken
sys.path.insert(0, REPO_DIR)

if os.path.exists(FINETUNE_TRAIN) and os.path.getsize(FINETUNE_TRAIN) > 0:
    n = len(np.memmap(FINETUNE_TRAIN, dtype=np.int32, mode='r'))
    print(f'✅ Déjà tokenisé : {n:,} tokens (int32)')
else:
    assert os.path.exists(PIAF_CACHE) and os.path.getsize(PIAF_CACHE) > 100, (
        'PIAF_CACHE introuvable. Lance la cellule 3.2 d\'abord.'
    )
    with open(PIAF_CACHE, encoding='utf-8') as f:
        piaf_data = json.load(f)

    enc = tiktoken.get_encoding('cl100k_base')

    def format_qa(ex):
        return (
            f"### Contexte\n{ex['context']}\n\n"
            f"### Question\n{ex['question']}\n\n"
            f"### Réponse\n{ex['answer']}<|endoftext|>"
        )

    print(f'Formatage et tokenisation de {len(piaf_data)} exemples...')
    all_tokens = []
    for ex in piaf_data:
        # allowed_special requis pour encoder <|endoftext|>
        toks = enc.encode(format_qa(ex), allowed_special={'<|endoftext|>'})
        all_tokens.extend(toks)

    arr     = np.array(all_tokens, dtype=np.int32)
    n_val   = max(500, int(len(arr)*0.05))
    n_train = len(arr) - n_val
    arr[:n_train].tofile(FINETUNE_TRAIN)
    arr[n_train:].tofile(FINETUNE_VAL)
    print(f'✅ Tokenisé : train={n_train:,} | val={n_val:,} tokens (int32)')

# Aperçu du format QA
with open(PIAF_CACHE, encoding='utf-8') as f:
    ex0 = json.load(f)[0]
print(f'\nFormat QA :\n{"─"*48}')
preview = format_qa(ex0) if 'format_qa' in dir() else (
    f"### Contexte\n{ex0['context'][:100]}...\n\n"
    f"### Question\n{ex0['question']}\n\n"
    f"### Réponse\n{ex0['answer']}<|endoftext|>"
)
print(preview[:350])
print('─'*48)


✅ Déjà tokenisé : 850,761 tokens (int32)

Format QA :
────────────────────────────────────────────────
### Contexte
Les dépenses des ménages représentent plus de 50 % de ces montants (14,2 milliards d'euros en 2003 e...

### Question
Combien de personnes travaillent au ministère des sports

### Réponse
100 000<|endoftext|>
────────────────────────────────────────────────


---
## 5️⃣ Pré-entraînement (Wikipedia FR)
> Entraîner le modèle depuis zéro. Les checkpoints périodiques appliquent
> un **rolling window de 2** : quand le 3e est sauvegardé, le 1er est supprimé.
> `best.pt` est toujours conservé séparément.

> 💡 Si Colab se déconnecte, relance juste la cellule 5.2 — la reprise est automatique.


In [12]:
# @title ⚙️ 5.1 — Configuration
import sys, torch, os, importlib
sys.path.insert(0, REPO_DIR)
import config

# --- Correction physique du fichier config.py ---
config_path = os.path.join(REPO_DIR, 'config.py')
with open(config_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open(config_path, 'w', encoding='utf-8') as f:
    for line in lines:
        if "vocab_size=50257" in line:
            line = line.replace("vocab_size=50257", "vocab_size=100277")
        f.write(line)

# Forcer le rechargement et la mise à jour manuelle des dictionnaires chargés
importlib.reload(config)
from config import TrainConfig, PRESETS

# Mise à jour manuelle pour garantir la prise en compte immédiate
for p in PRESETS:
    if PRESETS[p].vocab_size == 50257:
        PRESETS[p].vocab_size = 100277

PT_MAX_ITERS   = 10000  # @param {type:'integer'}
PT_BATCH       = 8      # @param {type:'slider',min:1,max:32,step:1}
PT_ACCUM       = 8      # @param {type:'slider',min:1,max:32,step:1}
PT_SEQ_LEN     = 512    # @param [256,512,1024]
PT_LR          = 3e-4   # @param {type:'number'}
PT_SAVE_EVERY  = 2000   # @param {type:'integer'}
PT_COMPILE     = False  # @param {type:'boolean'}

# Application de la config
mcfg = PRESETS[MODEL_SIZE]
mcfg.max_seq_len = PT_SEQ_LEN

pt_cfg               = TrainConfig()
pt_cfg.model_size    = MODEL_SIZE
pt_cfg.max_iters     = PT_MAX_ITERS
pt_cfg.batch_size    = PT_BATCH
pt_cfg.grad_accum    = PT_ACCUM
pt_cfg.seq_len       = PT_SEQ_LEN
pt_cfg.lr            = PT_LR
pt_cfg.min_lr        = PT_LR / 10
pt_cfg.warmup_iters  = max(200, PT_MAX_ITERS // 20)
pt_cfg.compile       = PT_COMPILE
pt_cfg.dtype         = 'float16'
pt_cfg.out_dir       = CKPT_PRETRAIN
pt_cfg.data_path     = PRETRAIN_TRAIN
pt_cfg.val_path      = PRETRAIN_VAL
pt_cfg.log_every     = 50
pt_cfg.eval_every    = 500
pt_cfg.save_every    = PT_SAVE_EVERY
pt_cfg.rolling_keep  = ROLLING_KEEP

batch_eff = PT_BATCH * PT_ACCUM
vram_gb   = torch.cuda.get_device_properties(0).total_memory/1e9
req_gb    = mcfg.count_params()*4*4/1e9

print(f'{"═"*56}')
print(f'  Pré-entraînement — MiniLLM-{MODEL_SIZE}')
print(f'{"═"*56}')
print(f'  Vocab size     : {mcfg.vocab_size}')
print(f'  Params         : {mcfg.count_params()/1e6:.1f}M')
print(f'  Itérations     : {PT_MAX_ITERS:,}')
print(f'  Batch effectif : {PT_BATCH} × {PT_ACCUM} = {batch_eff}')
print(f'  VRAM dispo     : {vram_gb:.1f} GB  |  Requis ~{req_gb:.1f} GB')
print(f'{"═"*56}')

════════════════════════════════════════════════════════
  Pré-entraînement — MiniLLM-50M
════════════════════════════════════════════════════════
  Vocab size     : 100277
  Params         : 83.5M
  Itérations     : 10,000
  Batch effectif : 8 × 8 = 64
  VRAM dispo     : 15.6 GB  |  Requis ~1.3 GB
════════════════════════════════════════════════════════


In [ ]:
# @title 🚀 5.2 — Lancer (reprise automatique si checkpoint existe)
import os, sys, torch, numpy as np
sys.path.insert(0, REPO_DIR)
from train import train

# --- Vérification d'intégrité des tokens ---
print("🔍 Vérification des tokens...")
def check_tokens(path, vocab_size):
    data = np.memmap(path, dtype=np.int32, mode='r')
    max_tok = data.max()
    if max_tok >= vocab_size:
        print(f"❌ ERREUR : Token ID {max_tok} trouvé dans {os.path.basename(path)}.")
        print(f"   Le modèle accepte max {vocab_size-1}. L'entraînement va planter.")
        return False
    return True

v_size = PRESETS[MODEL_SIZE].vocab_size
ok_tr = check_tokens(PRETRAIN_TRAIN, v_size)
ok_vl = check_tokens(PRETRAIN_VAL, v_size)

if not (ok_tr and ok_vl):
    raise ValueError("Les fichiers .bin contiennent des tokens hors limites pour ce modèle.")
print("✅ Tokens valides.")

# Nettoyage préventif
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Reprise automatique
best_ckpt = os.path.join(CKPT_PRETRAIN, 'best.pt')
if os.path.exists(best_ckpt):
    print(f'  ♻️  Reprise depuis : {best_ckpt}')
    pt_cfg.resume_from = best_ckpt
else:
    print('  🆕  Entraänement depuis zéro')
    pt_cfg.resume_from = ''

train(pt_cfg)
print(f'\n✅ Pré-entraänement terminé → {CKPT_PRETRAIN}/best.pt')


🔍 Vérification des tokens...
✅ Tokens valides.
  🆕  Entraänement depuis zéro

════════════════════════════════════════════════════════════
  MiniLLM v2 — Entraînement
════════════════════════════════════════════════════════════
  Appareil : cuda  |  Dtype : torch.float16
  Modèle   : MiniLLM-83M | layers=10 d_model=512 heads=8 kv=8 ffn=1408 ctx=512
  Params   : 83.47M
  Batch    : 8 × 8 accum = 64 effectif
  Rolling  : 2 checkpoint(s) numérotés gardés
────────────────────────────────────────────────────────────
Optimizer : 83,454,464 params avec weight_decay, 10,752 sans | fused=True

  Données train :
  → 335,028,176 tokens | ~81,793 batches
  Données val   :
  → 3,388,894 tokens | ~827 batches

════════════════════════════════════════════════════════════
  Début — 10,000 itérations
════════════════════════════════════════════════════════════



/content/minillm_v2/train.py:217: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler     = torch.cuda.amp.GradScaler(enabled=use_scaler)


iter      0 | val_loss 11.6162 | lr 6.0e-07 | 2642.7k tok/s | 6s
  ✅ Nouveau best : val_loss=11.6162 → /content/drive/MyDrive/MiniLLM_v2/checkpoints/pretrain/best.pt
  it     0 | loss 11.6103 | lr 6.0e-07 | tokens 0.0M
  it    50 | loss 10.3668 | lr 3.1e-05 | tokens 1.7M
  it   100 | loss 8.6531 | lr 6.1e-05 | tokens 3.3M
  it   150 | loss 6.9739 | lr 9.1e-05 | tokens 4.9M
  it   200 | loss 6.1496 | lr 1.2e-04 | tokens 6.6M
  it   250 | loss 5.8909 | lr 1.5e-04 | tokens 8.2M
  it   300 | loss 5.4167 | lr 1.8e-04 | tokens 9.9M
  it   350 | loss 5.4440 | lr 2.1e-04 | tokens 11.5M
  it   400 | loss 4.2950 | lr 2.4e-04 | tokens 13.1M
  it   450 | loss 3.5728 | lr 2.7e-04 | tokens 14.8M
iter    500 | val_loss 5.0192 | lr 3.0e-04 | 14.6k tok/s | 18.7min
  ✅ Nouveau best : val_loss=5.0192 → /content/drive/MyDrive/MiniLLM_v2/checkpoints/pretrain/best.pt
  it   500 | loss 4.1176 | lr 3.0e-04 | tokens 16.4M
  it   550 | loss 4.8872 | lr 3.0e-04 | tokens 18.1M
  it   600 | loss 4.8474 | lr 3.0e-0

In [ ]:
# @title 📈 5.3 — Courbe de loss
import torch, matplotlib.pyplot as plt, glob, os

iters, losses = [], []
for path in sorted(glob.glob(f'{CKPT_PRETRAIN}/*.pt')):
    try:
        c = torch.load(path, map_location='cpu', weights_only=False)
        if c.get('val_loss') and c.get('iter') is not None:
            iters.append(c['iter']); losses.append(c['val_loss'])
    except: pass

if losses:
    plt.figure(figsize=(10,4))
    plt.plot(iters, losses, 'b-o', lw=2, ms=4)
    plt.xlabel('Itération'); plt.ylabel('Val Loss')
    plt.title(f'MiniLLM-{MODEL_SIZE} — Pré-entraînement (Wikipedia FR)')
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{DRIVE_BASE}/pretrain_loss.png', dpi=150)
    plt.show()
    best_i = losses.index(min(losses))
    print(f'  Meilleure val_loss : {min(losses):.4f}  (iter {iters[best_i]:,})')
else:
    print('  Aucun checkpoint avec val_loss trouvé — lance 5.2 d\'abord.')


---
## 6️⃣ Fine-tuning QA (PIAF)
> Repart du meilleur checkpoint du pré-entraînement et se spécialise sur les 3 835 paires Q/R.
> Même logique de rolling checkpoints (2 max).


In [ ]:
# @title ⚙️ 6.1 — Configuration
import sys
sys.path.insert(0, REPO_DIR)
from config import TrainConfig, PRESETS
import os

FT_MAX_ITERS  = 3000   # @param {type:'integer'}
FT_BATCH      = 4      # @param {type:'slider',min:1,max:16,step:1}
FT_ACCUM      = 4      # @param {type:'slider',min:1,max:16,step:1}
FT_SEQ_LEN    = 512    # @param [256,512,1024]
FT_LR         = 5e-5   # @param {type:'number'}

pt_best = os.path.join(CKPT_PRETRAIN, 'best.pt')
assert os.path.exists(pt_best), (
    f'Checkpoint pré-entraînement introuvable : {pt_best}\n'
    f'Lance la section 5 d\'abord !'
)

ft_cfg               = TrainConfig()
ft_cfg.model_size    = MODEL_SIZE
ft_cfg.max_iters     = FT_MAX_ITERS
ft_cfg.batch_size    = FT_BATCH
ft_cfg.grad_accum    = FT_ACCUM
ft_cfg.seq_len       = FT_SEQ_LEN
ft_cfg.lr            = FT_LR
ft_cfg.min_lr        = FT_LR / 10
ft_cfg.warmup_iters  = 100
ft_cfg.compile       = True
ft_cfg.out_dir       = CKPT_FINETUNE
ft_cfg.data_path     = FINETUNE_TRAIN
ft_cfg.val_path      = FINETUNE_VAL
ft_cfg.resume_from   = pt_best
ft_cfg.log_every     = 50
ft_cfg.eval_every    = 300
ft_cfg.save_every    = 1000
ft_cfg.rolling_keep  = ROLLING_KEEP

mcfg = PRESETS[MODEL_SIZE]
mcfg.max_seq_len = FT_SEQ_LEN

print(f'{'═'*52}')
print(f'  Fine-tuning QA — MiniLLM-{MODEL_SIZE}')
print(f'{'═'*52}')
print(f'  Depuis         : {pt_best}')
print(f'  Dataset        : PIAF (3 835 paires Q/R françaises)')
print(f'  Itérations     : {FT_MAX_ITERS:,}')
print(f'  LR             : {FT_LR} (< LR pré-entraînement — fine-tuning)')
print(f'  Rolling keep   : {ROLLING_KEEP} checkpoints max')
print(f'{'═'*52}')


In [ ]:
# @title 🎯 6.2 — Lancer le fine-tuning
import sys
sys.path.insert(0, REPO_DIR)
from train import train

# Reprise si un fine-tuning était déjà en cours
import os
ft_best = os.path.join(CKPT_FINETUNE, 'best.pt')
if os.path.exists(ft_best):
    print(f'  ♻️  Reprise fine-tuning depuis : {ft_best}')
    ft_cfg.resume_from = ft_best

train(ft_cfg)
print(f'\n✅ Fine-tuning terminé → {CKPT_FINETUNE}/best.pt')


In [ ]:
# @title 📈 6.3 — Courbes comparatives
import torch, matplotlib.pyplot as plt, glob

fig, axes = plt.subplots(1, 2, figsize=(14,4))
configs = [
    (CKPT_PRETRAIN, f'Pré-entr. Wikipedia ({MODEL_SIZE})', 'steelblue'),
    (CKPT_FINETUNE, f'Fine-tuning PIAF ({MODEL_SIZE})',     'darkorange'),
]
for ax, (ckpt_dir, label, color) in zip(axes, configs):
    iters, losses = [], []
    for path in sorted(glob.glob(f'{ckpt_dir}/*.pt')):
        try:
            c = torch.load(path, map_location='cpu', weights_only=False)
            if c.get('val_loss') and c.get('iter') is not None:
                iters.append(c['iter']); losses.append(c['val_loss'])
        except: pass
    if losses:
        ax.plot(iters, losses, color=color, lw=2, marker='o', ms=3)
        ax.set_title(f'{label}\n(meilleure : {min(losses):.4f})')
        ax.set_xlabel('Itération'); ax.set_ylabel('Val Loss')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Pas de données', ha='center', transform=ax.transAxes)
        ax.set_title(label)
plt.suptitle(f'MiniLLM-{MODEL_SIZE} — Courbes d\'entraînement', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/all_losses.png', dpi=150)
plt.show()


---
## 7️⃣ Génération de texte
> Teste les deux modèles : **pré-entraîné** (complétion libre) et **fine-tuné** (QA).


In [ ]:
# @title 📂 7.1 — Charger un modèle
import os, sys
sys.path.insert(0, REPO_DIR)
from generate import load_model

MODELE = 'Fine-tuné QA'  # @param ['Pré-entraîné','Fine-tuné QA']

ckpt_path = (os.path.join(CKPT_FINETUNE, 'best.pt')
             if 'QA' in MODELE else
             os.path.join(CKPT_PRETRAIN, 'best.pt'))

assert os.path.exists(ckpt_path), (
    f'Checkpoint introuvable : {ckpt_path}\n'
    f'Lance la section correspondante d\'abord.'
)

gen_model, gen_cfg, gen_device = load_model(ckpt_path)
print(f'✅ Chargé : {MODELE}  ({gen_model.n_params/1e6:.1f}M params) sur {gen_device}')


In [ ]:
# @title ✍️ 7.2 — Complétion de texte libre
import tiktoken, sys
sys.path.insert(0, REPO_DIR)
from generate import generate

PROMPT      = 'La langue française est'  # @param {type:'string'}
MAX_TOKENS  = 150   # @param {type:'slider',min:50,max:400,step:25}
TEMPERATURE = 0.8   # @param {type:'slider',min:0.1,max:1.5,step:0.1}
TOP_K       = 50    # @param {type:'slider',min:0,max:100,step:5}
TOP_P       = 0.95  # @param {type:'slider',min:0.5,max:1.0,step:0.05}

enc    = tiktoken.get_encoding('cl100k_base')
tokens = enc.encode(PROMPT)
out    = generate(gen_model, tokens, MAX_TOKENS, TEMPERATURE, TOP_K, TOP_P, gen_device)
print(f'{'─'*50}')
print(enc.decode(out))
print(f'{'─'*50}')
print(f'({len(out)-len(tokens)} tokens générés)')


In [ ]:
# @title ❓ 7.3 — Mode Question / Réponse
# Fonctionne mieux avec le modèle fine-tuné (section 6)
import tiktoken, sys
sys.path.insert(0, REPO_DIR)
from generate import generate

CONTEXTE   = 'La Tour Eiffel est une tour en fer puddlé de 330 mètres de hauteur, '\
             'construite à Paris en 1889.'  # @param {type:'string'}
QUESTION   = 'Quelle est la hauteur de la Tour Eiffel ?'  # @param {type:'string'}
MAX_TOKENS_QA = 60  # @param {type:'slider',min:20,max:200,step:10}

enc    = tiktoken.get_encoding('cl100k_base')
prompt = (
    f'### Contexte\n{CONTEXTE}\n\n'
    f'### Question\n{QUESTION}\n\n'
    f'### Réponse\n'
)
tokens = enc.encode(prompt)
out    = generate(gen_model, tokens, MAX_TOKENS_QA,
                  temperature=0.3, top_k=20, top_p=0.9, device=gen_device)

answer = enc.decode(out[len(tokens):])
answer = answer.split('<|endoftext|>')[0].split('\n\n')[0].strip()

print(f'{'─'*50}')
print(f'Contexte  : {CONTEXTE}')
print(f'Question  : {QUESTION}')
print(f'{'─'*50}')
print(f'Réponse   : {answer}')
print(f'{'─'*50}')


---
## 8️⃣ Sauvegarde & rolling checkpoints
> Stratégie : **`best.pt` toujours gardé** + seulement les **2 derniers `ckpt_XXXXXX.pt`**.

> ```
> checkpoints/pretrain/
> ├── best.pt           ← toujours conservé (meilleure val_loss)
> ├── ckpt_004000.pt    ← avant-dernier   (gardé)
> └── ckpt_006000.pt    ← dernier         (gardé)
> # ckpt_002000.pt était là, mais a été supprimé automatiquement
> ```


In [ ]:
# @title 📁 8.1 — Résumé des fichiers sur Drive
import os, glob, torch

print(f'{'═'*58}')
print(f'  MiniLLM v2 — Fichiers sur Drive')
print(f'  {DRIVE_BASE}')
print(f'{'═'*58}')

for label, pattern in [
    ('Checkpoints pré-entraînement', f'{CKPT_PRETRAIN}/*.pt'),
    ('Checkpoints fine-tuning',      f'{CKPT_FINETUNE}/*.pt'),
    ('Données tokenisées',           f'{DATA_DIR}/**/*.bin'),
    ('Corpus texte',                 f'{DATA_DIR}/*.txt'),
]:
    files = sorted(glob.glob(pattern, recursive=True))
    files = [f for f in files if os.path.isfile(f)]
    if files:
        total_mb = sum(os.path.getsize(f) for f in files)/1e6
        print(f'\n  {label}  ({total_mb:.0f} MB total)')
        for f in files:
            size_mb = os.path.getsize(f)/1e6
            # Info iter/loss pour les checkpoints
            extra = ''
            if f.endswith('.pt'):
                try:
                    c = torch.load(f, map_location='cpu', weights_only=False)
                    it  = c.get('iter','?')
                    vl  = c.get('val_loss')
                    extra = f'iter={it}' + (f' val_loss={vl:.4f}' if vl else '')
                except: extra = '(illisible)'
            print(f'    {os.path.basename(f):28s} {size_mb:6.0f} MB  {extra}')

print(f'\n{'═'*58}')


In [ ]:
# @title 🔄 8.2 — Nettoyage manuel rolling (si nécessaire)
import os, glob
from train import _rolling_cleanup

DOSSIER = 'Pré-entraînement'  # @param ['Pré-entraînement','Fine-tuning']
GARDER  = 2                    # @param {type:'slider',min:1,max:5,step:1}

ckpt_dir = CKPT_PRETRAIN if DOSSIER == 'Pré-entraînement' else CKPT_FINETUNE

avant = sorted(glob.glob(f'{ckpt_dir}/ckpt_*.pt'))
print(f'Avant nettoyage ({len(avant)} ckpt numérotés) :')
for f in avant: print(f'  {os.path.basename(f)}')

n_removed = _rolling_cleanup(ckpt_dir, keep=GARDER)

apres = sorted(glob.glob(f'{ckpt_dir}/ckpt_*.pt'))
print(f'\nAprès ({len(apres)} ckpt) — {n_removed} supprimé(s) :')
for f in apres: print(f'  ✅ {os.path.basename(f)}')
if os.path.exists(os.path.join(ckpt_dir, 'best.pt')):
    print(f'  ✅ best.pt  (toujours conservé)')


In [ ]:
# @title 📥 8.3 — Télécharger un modèle sur ton PC
from google.colab import files
import os

TELECHARGER = 'Fine-tuné QA (best.pt)'  # @param ['Pré-entraîné (best.pt)','Fine-tuné QA (best.pt)']

path = (os.path.join(CKPT_FINETUNE, 'best.pt') if 'QA' in TELECHARGER
        else os.path.join(CKPT_PRETRAIN, 'best.pt'))

if os.path.exists(path):
    size_mb = os.path.getsize(path)/1e6
    print(f'Téléchargement : {os.path.basename(path)}  ({size_mb:.0f} MB)...')
    files.download(path)
else:
    print(f'❌ Fichier introuvable : {path}')
    print(f'   Lance la section 5 (pré-entraînement) ou 6 (fine-tuning) d\'abord.')


In [ ]:
# @title 📤 8.4 — Uploader un checkpoint (reprendre sur Colab)
from google.colab import files
import shutil, os

DESTINATION = 'Pré-entraînement'  # @param ['Pré-entraînement','Fine-tuning']
dest_dir = CKPT_PRETRAIN if DESTINATION == 'Pré-entraînement' else CKPT_FINETUNE

print('Sélectionne ton fichier best.pt...')
uploaded = files.upload()
for fname in uploaded:
    dest = os.path.join(dest_dir, 'best.pt')
    shutil.move(fname, dest)
    print(f'✅ Uploadé → {dest}')
print('Tu peux maintenant relancer la section 5 ou 6 (reprise automatique).')


---
## 🎁 Conseils & prochaines étapes

### Améliorer les performances

| Action | Impact estimé |
|--------|---------------|
| `WIKI_FRACTION = '20%'` → plus de texte | +++++ |
| Augmenter `PT_MAX_ITERS` | ++++ |
| Passer à `MODEL_SIZE = '125M'` | +++ |
| Ajouter un corpus Fulfulde (projet Tardigrade) | +++ |
| Entraîner un tokenizer custom fr+fuv | ++ |

### Intégrer tes données Fulfulde
```python
# Dans la cellule 3.1, après le téléchargement Wikipedia :
import os
FULFULDE_FILE = '/content/drive/MyDrive/corpus_fulfulde.txt'   # ton corpus
if os.path.exists(FULFULDE_FILE):
    with open(WIKI_CACHE, 'a', encoding='utf-8') as out:
        with open(FULFULDE_FILE, encoding='utf-8') as f:
            out.write('\n\n' + f.read())
    print('✅ Corpus Fulfulde ajouté à Wikipedia FR')
```

### Liens
- 📦 Repo : https://github.com/bono-p/minillm_v2
- 📚 PIAF : https://huggingface.co/datasets/etalab-ia/piaf
- 🌐 Wikipedia : https://huggingface.co/datasets/wikimedia/wikipedia
